# 1. Validation label 상태 및 특이 사례 분석
정선우 · 데이터/평가/통합

목적: 300개 validation label 확인, label 품질 확인, 특이 날짜 사례 파악, 이후 OCR/parser 평가 준비.

기존 사람이 작성한 label과 notes만 분석한다. OCR 및 날짜 parser는 구현하지 않는다. 일부 요소가 확인되지 않으면 해당 요소만 NONE인 것이 정상 정답일 수 있다 (2026-01-NONE, NONE-04-08). 분석에서만 NONE 대소문자를 동일하게 취급한다. 제품의 YMD/DMY/MDY 표기 순서나 지역을 추정하지 않는다. 원본 CSV는 읽기만 한다.

## 2. 라이브러리 및 경로 설정
표준 라이브러리와 pandas를 사용한다. pandas가 있는 커널에서 실행한다. 루트 또는 notebooks/에서 실행할 수 있도록 상위 폴더를 탐색한다.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

cwd = Path.cwd().resolve()
csv_path = next((root / "labels" / "labels_300.csv" for root in (cwd, *cwd.parents)
                 if (root / "labels" / "labels_300.csv").is_file()), None)
if csv_path is None:
    raise FileNotFoundError("프로젝트 루트 또는 notebooks/에서 실행하세요: labels/labels_300.csv 없음")
original_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
print("입력:", csv_path)


입력: C:\Users\jsw58\Desktop\itda-ocr\labels\labels_300.csv


## 3. 데이터 로드
모든 컬럼을 문자열로 읽고 NA 자동 인식을 끈다. image_id의 기존 앞자리 0과 추가 컬럼을 보존한다. 없는 0을 추가하지 않는다.

In [2]:
labels = pd.read_csv(csv_path, dtype=str, encoding="utf-8-sig", keep_default_na=False, na_filter=False)
required = ["file_name", "image_id", "year", "month", "day", "final_date", "notes"]
missing = [c for c in required if c not in labels.columns]
if missing:
    raise ValueError(f"필수 컬럼 누락: {missing}")
original_labels = labels.copy(deep=True)
from IPython.display import Markdown, display


def display_review_table(frame):
    # Bound each display and wrap full text inside the available notebook width.
    page_size = 60
    for start in range(0, max(len(frame), 1), page_size):
        page = frame.iloc[start:start + page_size]
        if len(frame) > page_size:
            print(f"{start + 1}-{start + len(page)} / {len(frame)}")
        with pd.option_context("display.max_rows", page_size,
                               "display.max_columns", 9,
                               "display.max_colwidth", 120):
            display(page.style.hide(axis="index").format(escape="html")
                    .set_table_attributes('style="width:100%; table-layout:fixed;"')
                    .set_properties(**{"white-space": "normal",
                                       "overflow-wrap": "anywhere",
                                       "text-align": "left",
                                       "vertical-align": "top"})
                    .set_table_styles([
                        {"selector": "th", "props": [("white-space", "normal"),
                                                       ("overflow-wrap", "anywhere"),
                                                       ("text-align", "left")]},
                    ] + [
                        {"selector": f"th.col{page.columns.get_loc(column)}",
                         "props": [("width", width)]}
                        for column, width in [("notes", "35%"), ("case_type", "25%")]
                        if column in page.columns
                    ]))


display_review_table(labels.loc[:, required].head())


file_name,image_id,year,month,day,final_date,notes
000018.jpg,18,2026,4,24,2026-04-24,
000026.jpg,26,2025,11,13,2025-11-13,
000030.jpg,30,2028,08,17,2028-08-17,
000060.jpg,60,2026,04,30,2026-04-30,
000075.jpg,75,2026,01,06,2026-01-06,


## 4. 기본 검증
공백만 있는 값도 빈 값으로 센다. NONE은 빈 값이 아니다. 중복은 공백을 제거한 비어 있지 않은 ID에서 첫 행 이후 반복 행 수다. 원본 값은 수정하지 않는다.

In [3]:
ids = labels["image_id"].str.strip()
checks = {"전체 행 수": len(labels),
          "image_id 중복 개수 (첫 행 제외)": int(ids[ids.ne("")].duplicated().sum()),
          "빈 image_id 개수": int(ids.eq("").sum())}
checks.update({f"{c} 빈 값 개수": int(labels[c].str.strip().eq("").sum())
               for c in ["year", "month", "day", "final_date"]})
print("컬럼 목록:", labels.columns.tolist())
print(pd.Series(checks).to_string())
print("300행 확인:", "일치" if len(labels) == 300 else f"확인 필요 ({len(labels)}행)")


컬럼 목록: ['file_name', 'image_id', 'year', 'month', 'day', 'final_date', 'notes', 'Unnamed: 7']
전체 행 수                     300
image_id 중복 개수 (첫 행 제외)      0
빈 image_id 개수                0
year 빈 값 개수                  0
month 빈 값 개수                 0
day 빈 값 개수                   0
final_date 빈 값 개수            0
300행 확인: 일치


## 5. 부분 NONE 분석
분석용 마스크에서만 공백과 대소문자를 정규화한다. 부분 NONE은 1~2개 요소가 NONE인 행이다. 모두 NONE인 행과 구분하며 NONE을 오류로 판단하지 않는다.

In [4]:
fields = ["year", "month", "day"]
none_mask = labels[fields].apply(lambda s: s.str.strip().str.casefold().eq("none"))
none_count = none_mask.sum(axis=1)
partial_none = none_count.between(1, 2)
print("요소별 NONE 수:")
print(none_mask.sum().to_string())
none_types = {f"{c}만 NONE": int((none_mask[c] & none_count.eq(1)).sum()) for c in fields}
none_types.update({"2개 field가 NONE": int(none_count.eq(2).sum()),
                   "year/month/day 모두 NONE": int(none_count.eq(3).sum())})
print(pd.Series(none_types).to_string())


요소별 NONE 수:
year      4
month     0
day      14
year만 NONE                 4
month만 NONE                0
day만 NONE                 14
2개 field가 NONE             0
year/month/day 모두 NONE     0


## 6. notes 분석
키워드 기반 1차 후보 분류이며 중복을 허용한다. 확정 사실이나 오류 판정이 아니다. 거꾸로는 이미지 방향일 수도 있어 실제 날짜 순서를 확정하지 않는다. 영문 월은 JUL2023처럼 숫자와 붙은 토큰도 포함한다.

복수 날짜는 명시된 날짜 개수, notes 안의 날짜 모양 문자열 2개 이상, 제조/기한 동시 언급을 탐색한다. 단순히 숫자 두 개라는 표현만으로 복수 날짜로 추정하지 않는다. 제조/기한 동시 언급은 부정문에서도 잡힐 수 있어 원문 검토가 필요하다. 날짜 모양 탐색은 notes 분류에만 쓰며 날짜 값이나 순서를 파싱하지 않는다. 기타 notes는 아래 유형에 해당하지 않는 비어 있지 않은 notes다.

In [5]:
notes = labels["notes"].str.strip()
notes_present = notes.ne("")
def contains(pattern):
    return notes.str.contains(pattern, case=False, regex=True, na=False)

order_pattern = r"거꾸로|반대로|역순|날짜\s*순서|YMD|DMY|MDY|DD[\s./-]*MM[\s./-]*YY|MM[\s./-]*DD[\s./-]*YY|YY(?:YY)?[\s./-]*MM[\s./-]*DD|(?:월\s*[-/]?\s*일|일\s*[-/]?\s*월)\s*[-/]?\s*[연년]"
month_pattern = r"(?<![A-Za-z])(?:JAN(?:UARY)?|FEB(?:RUARY)?|MAR(?:CH)?|APR(?:IL)?|MAY|JUN(?:E)?|JUL(?:Y)?|AUG(?:UST)?|SEP(?:TEMBER)?|SEPT|OCT(?:OBER)?|NOV(?:EMBER)?|DEC(?:EMBER)?)(?![A-Za-z])|영문\s*월|월[^,\n]{0,15}영어|영어\s*\d{1,2}월"
manufacture = contains(r"제조|(?<![A-Za-z])(?:PRD|MFG|MFD)(?![A-Za-z])|PRO\.?\s*DATE|production\s*date")
expiry = contains(r"소비기한|유통기한|까지|(?<![A-Za-z])(?:EXP(?:IRY|IRATION)?|BB|BBD)(?![A-Za-z])|best\s*(?:before|by)")
coexist = manufacture & expiry
explicit_multiple = contains(r"(?:날짜|날자)(?:가)?\s*(?:두|세|[2-9])\s*개|복수\s*날짜|여러\s*(?:개.*)?날짜")
date_mentions = notes.str.count(r"(?<!\d)\d{2,4}[./-]\d{1,2}[./-]\d{1,4}(?!\d)")
case_flags = pd.DataFrame({
    "날짜 순서 관련 후보": contains(order_pattern),
    "영문 월 관련 후보": contains(month_pattern),
    "복수 날짜 관련 후보": explicit_multiple | date_mentions.ge(2) | coexist,
    "제조일/소비기한 동시 존재 후보": coexist,
    "EXP / EXPIRY 관련 후보": contains(r"(?<![A-Za-z])EXP(?:IRY|IRATION)?(?![A-Za-z])"),
    "유통기한 / 소비기한 관련 후보": contains(r"유통기한|소비기한"),
    "일부 날짜 요소 부재 notes 후보": contains(r"(?:연도|년도|연|년|월|일|YEAR|MONTH|DAY)(?:이|가|은|는)?\s*(?:없|누락|미표기|생략)|일부.*(?:없|누락)|(?:연도|년도|YEAR|MONTH|DAY)\s*확인\s*불가"),
}, index=labels.index)
case_flags["기타 notes"] = notes_present & ~case_flags.any(axis=1)
case_flags.sum().rename("후보 수 (중복 가능)").to_frame()


,후보 수 (중복 가능)
날짜 순서 관련 후보,52
영문 월 관련 후보,16
복수 날짜 관련 후보,17
제조일/소비기한 동시 존재 후보,7
EXP / EXPIRY 관련 후보,10
유통기한 / 소비기한 관련 후보,22
일부 날짜 요소 부재 notes 후보,8
기타 notes,75


## 7. 특이 사례 테이블
notes 후보와 label의 NONE 사례를 합친다. case_type은 내부 복사본에만 추가한다. 부분/전체 NONE은 정상 정답일 수 있다.

주요 5개 유형을 개별 표로 표시하며, 중복 해당 이미지는 여러 표에 나타난다. 긴 notes와 case_type은 셀 안에서 줄바꿈하고, 60건을 넘는 표는 나누어 표시한다. 기타 notes는 개수만 표시하며 `other_notes`에서 별도로 확인할 수 있다. 전체 후보는 `special_cases`에 유지한다.

In [6]:
table_flags = case_flags.copy()
table_flags["부분 NONE (정상 정답 가능)"] = partial_none
table_flags["모두 NONE (정상 정답 가능)"] = none_count.eq(3)
selected = table_flags.any(axis=1)
special_cases = labels.loc[selected, required].copy()
special_cases["case_type"] = table_flags.loc[selected].apply(
    lambda row: " | ".join(row.index[row].tolist()), axis=1)

review_columns = ["file_name", "image_id", "final_date", "notes", "case_type"]
review_groups = [
    ("부분 NONE", partial_none, ["file_name", "image_id", "year", "month", "day",
                               "final_date", "notes", "case_type"]),
    *[(name, case_flags[name], review_columns) for name in [
        "날짜 순서 관련 후보", "영문 월 관련 후보", "복수 날짜 관련 후보",
        "제조일/소비기한 동시 존재 후보"]],
]
for name, mask, columns in review_groups:
    display(Markdown(f"### {name}: {int(mask.sum())}건"))
    display_review_table(special_cases.loc[mask.reindex(special_cases.index), columns])

other_notes = special_cases.loc[
    case_flags["기타 notes"].reindex(special_cases.index), review_columns].copy()
print(f"기타 notes: {len(other_notes)}건 (별도 확인: other_notes)")


### 부분 NONE: 18건

file_name,image_id,year,month,day,final_date,notes,case_type
000515.jpg,515,2026,01,NONE,2026-01-none,,부분 NONE (정상 정답 가능)
000763.jpg,763,NONE,11,12,none-11-12,,부분 NONE (정상 정답 가능)
000789.jpeg,789,NONE,12,19,none-12-19,,부분 NONE (정상 정답 가능)
001173.jpg,1173,2021,09,NONE,2021-09-none,,부분 NONE (정상 정답 가능)
001205.jpg,1205,2021,05,NONE,2021-05-none,,부분 NONE (정상 정답 가능)
002062.jpg,2062,2023,02,NONE,2023-02-none,,부분 NONE (정상 정답 가능)
002278.jpg,2278,2023,07,NONE,2023-07-none,JUL2023,영문 월 관련 후보 | 부분 NONE (정상 정답 가능)
002480.jpg,2480,2022,07,NONE,2022-07-none,,부분 NONE (정상 정답 가능)
003004.jpg,3004,2023,05,NONE,2023-05-NONE,,부분 NONE (정상 정답 가능)
000907.jpg,907,2022,01,NONE,2022-01-NONE,"DAY가 없음, 날짜 거꾸로",날짜 순서 관련 후보 | 일부 날짜 요소 부재 notes 후보 | 부분 NONE (정상 정답 가능)


### 날짜 순서 관련 후보: 52건

file_name,image_id,final_date,notes,case_type
000715.jpg,715,2026-06-26,"DD/MM/YYYY, 흐림, 반사있음",날짜 순서 관련 후보
000820.jpeg,820,2026-05-18,DD/MM/YYYY,날짜 순서 관련 후보
001075.jpg,1075,2021-09-30,DD/MM/YYYY,날짜 순서 관련 후보
001175.jpg,1175,2021-09-07,DD/MM/YYYY,날짜 순서 관련 후보
001683.jpg,1683,2021-11-14,DD/MM/YYYY,날짜 순서 관련 후보
001857.jpg,1857,2022-07-17,DD/MM/YYYY,날짜 순서 관련 후보
002019.jpg,2019,2022-03-09,DD/MM/YYYY,날짜 순서 관련 후보
002162.jpg,2162,2023-06-18,MM/DD/YY,날짜 순서 관련 후보
002443.jpg,2443,2021-11-11,DD/MM/YY,날짜 순서 관련 후보
002492.jpg,2492,2022-08-09,DD.MM.YY,날짜 순서 관련 후보


### 영문 월 관련 후보: 16건

...

### 복수 날짜 관련 후보: 17건

file_name,image_id,final_date,notes,case_type
000245.jpg,245,2026-02-24,2025-09-25부터 2026-02-24까지,복수 날짜 관련 후보
000853.jpeg,853,2026-09-24,25.09.25-26.09.24,복수 날짜 관련 후보
000957.jpg,957,2021-02-22,21.01.29제조 21.02.22까지,복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보
000007.jpg,7,2026-06-25,"~부터, ~까지 라고 적혀있어서 날짜가 2개 나옴, 클로드도 25일인데 26으로 읽음",복수 날짜 관련 후보
000148.jpg,148,2026-02-26,"EXP, PRD 같이 있음",복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | EXP / EXPIRY 관련 후보
000250.jpg,250,2026-03-04,날짜 두 개 같이 있음,복수 날짜 관련 후보
000378.jpg,378,2026-05-29,날짜가 두 개인데 게다가 거꾸로,날짜 순서 관련 후보 | 복수 날짜 관련 후보
000384.jpg,384,2026-07-08,"날자 두 개, 글씨 작음",복수 날짜 관련 후보
000400.jpg,400,2026-01-11,"날짜 두 개, 뒤에 C 붙어있음",복수 날짜 관련 후보
000422.jpg,422,2026-06-09,"날짜 두개, 까지가 적혀 있긴 한데 연함",복수 날짜 관련 후보


### 제조일/소비기한 동시 존재 후보: 7건

file_name,image_id,final_date,notes,case_type
000957.jpg,957,2021-02-22,21.01.29제조 21.02.22까지,복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보
000148.jpg,148,2026-02-26,"EXP, PRD 같이 있음",복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | EXP / EXPIRY 관련 후보
000954.jpg,954,2021-02-05,"숫자 두 개 있고, 뒤에 B, 그리고 날짜가 약간 내려가서 까지 옆에 소비기한이 아닌 제조기한이 있음",복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | 유통기한 / 소비기한 관련 후보
001097.jpg,1097,2021-06-11,"숫자 두개 있고, 유통기한이며, 프린팅 문제인지 소비기한 바로 옆에 '부터'가 써져있음 (날짜는 제조기한 아래에)",복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | 유통기한 / 소비기한 관련 후보
001141.jpg,1141,2022-03-14,"날짜 두 개고 거꾸로 써져있으며 PRO.DATE, EXP.DATE로 적혀 있음 배경 빨간색",날짜 순서 관련 후보 | 복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | EXP / EXPIRY 관련 후보
001869.jpg,1869,2021-09-26,"특이하게 제조보다 소비기한이 위에 있고 EXP으로 포기, 그리고 거꾸로",날짜 순서 관련 후보 | 복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보 | EXP / EXPIRY 관련 후보 | 유통기한 / 소비기한 관련 후보
003228.jpg,3228,2022-04-26,"거꾸로, BB 옆에 적혀 있음. 제조기한보다 위에 적혀 있음, 심지어 Pack date도 있어서 날짜가 3개임",날짜 순서 관련 후보 | 복수 날짜 관련 후보 | 제조일/소비기한 동시 존재 후보


기타 notes: 75건 (별도 확인: other_notes)


## 8. 요약
이미지 수는 label 행 기준이며 고유 ID 수도 함께 표시한다. 후보는 중복 가능하다. 이후 평가에서도 요소별 NONE을 보존하고 후보 원문을 검토한다. 실행 출력에는 비공개 label과 notes가 포함되므로 공유/커밋 전 출력을 지운다.

In [7]:
assert labels.equals(original_labels), "원본 DataFrame 변경 감지"
assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == original_sha256, "입력 CSV 변경 감지"
summary = {"전체 validation 이미지 수 (label 행 기준)": len(labels),
           "고유 image_id 수 (빈 값 제외)": int(ids[ids.ne("")].nunique()),
           "부분 NONE 사례 수 (1~2개 요소)": int(partial_none.sum()),
           "notes가 있는 사례 수": int(notes_present.sum())}
for name in ["날짜 순서 관련 후보", "영문 월 관련 후보", "복수 날짜 관련 후보", "제조일/소비기한 동시 존재 후보"]:
    summary[f"{name} 수"] = int(case_flags[name].sum())
print(pd.Series(summary).to_string())
print("원본 CSV 및 로드한 DataFrame 변경 없음")


전체 validation 이미지 수 (label 행 기준)    300
고유 image_id 수 (빈 값 제외)              300
부분 NONE 사례 수 (1~2개 요소)               18
notes가 있는 사례 수                      174
날짜 순서 관련 후보 수                        52
영문 월 관련 후보 수                         16
복수 날짜 관련 후보 수                        17
제조일/소비기한 동시 존재 후보 수                   7
원본 CSV 및 로드한 DataFrame 변경 없음
